---
title: "Estimating SNP heritability"
format:
  html:
    code-fold: false
    toc: true
  ipynb:
    toc: true
    number-sections: false
bibliography: references/references_5.bib
---

::: {.callout-note title="Important notes for this notebook" icon=false}

Before moving to advanced mixed-model GWAS (`GWAS9`), fine-mapping (`GWAS10`), and Bayesian polygenic prediction (`GWAS11`), we need to understand **SNP heritability ($h^2_{\text{SNP}}$)**. Heritability quantifies the proportion of phenotypic variance explained by measured genetic variants and acts as a key tuning parameter in almost all advanced statistical genetics methods. In this tutorial, we explore the assumptions behind different heritability models (such as GCTA vs. LDAK) and estimate SNP heritability from both **individual-level genotypes** (`ldak --reml`) and **GWAS summary statistics** (`ldak --sum-hers`).

## Learning outcomes

- **Define** narrow-sense SNP heritability ($h^2_{\text{SNP}}$) and explain how it bounds polygenic prediction accuracy
- **Compare** the genetic architecture assumptions of the **GCTA** ($\alpha = -1$) and **LDAK** ($\alpha = -0.25$) heritability models
- **Estimate** SNP heritability from individual-level genotype data using REML (`ldak --reml`)
- **Estimate** SNP heritability from GWAS summary statistics using SumHer (`ldak --sum-hers`) and explain how LD tagging separates polygenicity from confounding

## How to make this notebook work

* In this notebook, we will use both `bash command line` and `R` programming languages. Remember to **change the kernel** whenever you transition from one language to the other (`Kernel --> Change Kernel`) indicated by the languages' images. We will first run `Bash` commands.  
  
<img src="Images/bash.png" alt="Bash" width="80"> Choose the Bash kernel
 
:::

:::{.callout-note title="Some theory"}

### Intuition First: How can we estimate heritability without knowing which SNPs are causal?

In `GWAS5`, we saw that most individual SNPs have tiny effects that do not pass the strict genome-wide significance threshold ($p < 5 \times 10^{-8}$). How, then, can we estimate the **total** fraction of trait variance explained by all measured SNPs simultaneously?

- **With individual-level data (`REML`)**: Even among "unrelated" people from the same population, some pairs of individuals happen by chance to share slightly more alleles across the genome than other pairs (e.g., $+0.5\%$ vs. $-0.5\%$ genome-wide similarity $K_{ij}$). If a trait like height is heritable, pairs of people who are slightly more genetically similar across the genome will also have slightly more similar heights ($y_i \cdot y_j$)! Measuring how strongly phenotypic similarity tracks genome-wide genetic similarity (`GRM`) gives us **SNP heritability ($h^2_{\text{SNP}}$)**.
- **With GWAS summary statistics (`SumHer`)**: Suppose you only have marginal GWAS $p$-values and no individual genotypes. A SNP sitting inside a large Linkage Disequilibrium (LD) block "tags" (is correlated with) many neighboring variants, so it is much more likely to tag a causal variant than an isolated SNP in a low-LD region. Regressing the GWAS $\chi^2$ test statistics on each SNP's **LD tagging score** reveals the total polygenic heritability.
- **Why weight SNPs by LD and MAF (`GCTA` vs. `LDAK`)?**: Imagine **5 witnesses in an LD block** who all saw the exact same event and give identical testimony ($r^2 \approx 0.99$). Counting all 5 as independent witnesses (`GCTA`) over-counts that region 5-fold! Thinning redundant witnesses and weighting variants by allele frequency (`LDAK-Thin`) prevents LD double-counting.

### Formal Definition of SNP Heritability ($h^2_{\text{SNP}}$)

Suppose we model a quantitative phenotype $y$ across all $M$ measured SNPs simultaneously:

$$y = Z\gamma + \sum_{j=1}^{M} X_j b_j + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$$

where $Z\gamma$ represents fixed covariates (such as sex and principal components), $X_j$ is the genotype dosage vector ($0, 1, or 2$ copies of allele `A1`) for SNP $j$ with minor allele frequency $f_j$, and $b_j$ is its per-allele additive effect. Treating the total genetic value $g = \sum_{j=1}^M X_j b_j$ as a random effect with variance $\sigma_g^2$, the **SNP heritability** is the proportion of phenotypic variance attributable to all measured SNPs:

$$h^2_{\text{SNP}} = \frac{\sigma_g^2}{\sigma_g^2 + \sigma_e^2}$$

Notice that $h^2_{\text{SNP}}$ is always smaller than classic twin/pedigree narrow-sense heritability ($h^2$), because $h^2_{\text{SNP}}$ only captures genetic variation tagged by the SNPs present on our genotyping array or imputation panel.

### The Heritability Model: How does heritability vary across SNPs?

Under Hardy-Weinberg equilibrium, the variance of genotype $X_j$ is $2 f_j (1 - f_j)$, so the heritability contributed by SNP $j$ is $h_j^2 = 2 f_j (1 - f_j) b_j^2$. To construct a Genetic Relatedness Matrix (GRM) or set priors on SNP effects, every method must assume a **heritability model** for the expected heritability per SNP, $\mathbb{E}[h_j^2]$.

The general **LDAK heritability model** ([Speed et al., 2012](https://doi.org/10.1016/j.ajhg.2012.10.010); [Speed et al., 2017](https://doi.org/10.1038/ng.3865)) writes this as:

$$\mathbb{E}[h_j^2] \propto \left[ 2 f_j (1 - f_j) \right]^{1 + \alpha} \times w_j$$

- **$\alpha$ (MAF scaling parameter)**: Controls how per-allele effect sizes $b_j$ vary with minor allele frequency $f_j$. Strong negative (purifying) natural selection prevents variants with huge effects from becoming common, yet common variants still vary across many more people ($2 f_j (1 - f_j)$ is largest at $f_j = 0.5$).
- **$w_j$ (LD weight)**: Downweights or thins SNPs in high-LD blocks so that regions with dozens of redundant, correlated SNPs are not double-counted.

| Heritability Model | MAF Parameter ($\alpha$) | LD Weighting ($w_j$) | Key Assumption |
|---|---|---|---|
| **GCTA Model** | $\alpha = -1$ | $w_j = 1$ (None) | Every SNP is expected to explain the **exact same heritability** ($\mathbb{E}[h_j^2] = \text{constant}$), regardless of allele frequency or LD. High-LD regions contribute more heritability simply because they contain more SNPs. |
| **LDAK-Thin Model** (Human Default) | $\alpha = -0.25$ | $w_j \in \{0, 1\}$ via LD thinning | Common SNPs explain more heritability per variant than rare SNPs ($\mathbb{E}[h_j^2] \propto [2 f_j(1-f_j)]^{0.75}$), matching empirical human traits. Duplicate SNPs ($r^2 > 0.98$) are pruned. |
| **BLD-LDAK Model** (Summary stats) | $\alpha = -0.25$ | Continuous LD weights + 65 functional annotations | Allows coding, conserved, and regulatory regions to be enriched for heritability alongside MAF and LD scaling. |

### Why Heritability Matters for the Rest of the Course

1. **Mixed-Model GWAS (`GWAS9`, e.g., LDAK-KVIK, BOLT-LMM)**: Uses $\hat{h}^2_{\text{SNP}}$ and $\mathbb{E}[h_j^2]$ in Step 1 to regularize the whole-genome regression model that predicts the polygenic background $\hat{g}^{(-c)}$ on each chromosome.
2. **Fine-Mapping (`GWAS10`, e.g., SuSiE)**: Uses per-locus heritability to calibrate the prior variance of causal effect sizes.
3. **Advanced Polygenic Scores (`GWAS11`, e.g., MegaPRS, LDpred2, LDAK-Predict)**: Uses $\hat{h}^2_{\text{SNP}}$ and per-SNP heritabilities $\mathbb{E}[h_j^2]$ to determine how strongly noisy GWAS effect estimates are shrunk toward zero.

> **Quick Concept Check (from Lecture 8)**:
> - *If a trait has Twin $h^2 = 80\%$ and Array $h^2_{\text{SNP}} = 45\%$, what is the maximum possible $R^2$ of a standard SNP-chip Polygenic Risk Score?* **$45\%$ ($R^2 \le h^2_{\text{SNP}}$)**, because a SNP chip can only predict variation tagged by its measured SNPs.
> - *Why do we run `ldak --thin` before computing the Kinship Matrix $K$?* Without thinning, an LD block with **5 identical SNPs ($r^2 = 1.00$)** is counted **5$\times$ too heavily** in $K$!

:::

# Individual-Level Heritability Estimation (`ldak --reml`)

When we have individual-level genotypes and phenotypes, we estimate $h^2_{\text{SNP}}$ using **Restricted Maximum Likelihood (REML)**:
1. Compute the $N \times N$ **Genetic Relatedness Matrix (GRM)** $K$ across all pairs of individuals (`ldak --calc-kins-direct`).
2. Fit the genetic variance $\sigma_g^2$ and environmental variance $\sigma_e^2$ by maximizing the restricted likelihood of $y \sim \mathcal{N}(Z\gamma, \sigma_g^2 K + \sigma_e^2 I)$ (`ldak --reml`).

<img src="Images/bash.png" alt="Bash" width="40"> Switch to the Bash kernel.

First, let's link the `Data` folder, create `Results/GWAS8`, and strip the header lines from the height phenotype (`Data/EUR.height`) and covariates (`Data/EUR.covariate`) so `LDAK` can read them:

In [ ]:
ln -sf ../Data
mkdir -p Results/GWAS8

# Strip header lines from phenotype and covariate files for compatibility
awk 'NR>1 {print $1, $2, $3}' Data/EUR.height > Results/GWAS8/EUR.pheno
awk 'NR>1 {print $0}' Data/EUR.covariate > Results/GWAS8/EUR.covar

head -n 5 Results/GWAS8/EUR.pheno
head -n 5 Results/GWAS8/EUR.covar

Now let's compute the GRM and run REML under both the **GCTA model** (`--power -1`, no LD weights) and the **LDAK-Thin model** (`--thin` with $r^2 > 0.98$, `--power -0.25`).

Key `LDAK` flags used below:
- `--thin <prefix>`: Prunes out near-duplicate SNPs ($r^2 > 0.98$ within 100 kb windows) and writes the retained SNP list to `<prefix>.in`.
- `--calc-kins-direct <prefix>`: Directly computes the $N \times N$ Genetic Relatedness Matrix (GRM) from the PLINK `--bfile`.
- `--power <alpha>`: Sets the MAF scaling parameter $\alpha$ (`-1` for the GCTA model; `-0.25` for the human default LDAK model).
- `--weights <file>` / `--ignore-weights YES`: Tells `LDAK` whether to weight SNPs by LD thinning or give all SNPs weight 1.
- `--reml <prefix>`: Runs Restricted Maximum Likelihood on `--grm`, `--pheno`, and `--covar` to estimate SNP heritability.

In [ ]:
# 1. GCTA Model (power = -1, uniform expected heritability per SNP)
ldak --calc-kins-direct Results/GWAS8/kins_gcta \
     --bfile Data/EUR.QC \
     --ignore-weights YES \
     --power -1

ldak --reml Results/GWAS8/reml_gcta \
     --grm Results/GWAS8/kins_gcta \
     --pheno Results/GWAS8/EUR.pheno \
     --covar Results/GWAS8/EUR.covar

In [ ]:
# 2. LDAK-Thin Model (thin duplicate SNPs with r^2 > 0.98, then compute GRM with power = -0.25)
ldak --thin Results/GWAS8/thin \
     --bfile Data/EUR.QC \
     --window-prune 0.98 \
     --window-kb 100

awk '{print $1, 1}' Results/GWAS8/thin.in > Results/GWAS8/weights.thin

ldak --calc-kins-direct Results/GWAS8/kins_ldak \
     --bfile Data/EUR.QC \
     --weights Results/GWAS8/weights.thin \
     --power -0.25

ldak --reml Results/GWAS8/reml_ldak \
     --grm Results/GWAS8/kins_ldak \
     --pheno Results/GWAS8/EUR.pheno \
     --covar Results/GWAS8/EUR.covar

# Summary-Statistics Heritability Estimation (`ldak --sum-hers`)

When our individual-level sample size is small ($N \approx 500$ in `Data/EUR.QC`), REML heritability estimates have large standard errors ($\text{SD}(\hat{h}^2) \approx \frac{316}{N}$). However, if we have summary statistics from a large GWAS meta-analysis—such as `Data/Height.QC.gz` ($N = 253,288$)—we can estimate $h^2_{\text{SNP}}$ accurately using **SumHer** (`ldak --sum-hers`).

Under a polygenic model, a SNP's expected marginal GWAS $\chi^2$ test statistic is linearly related to the total heritability it tags through LD:

$$\mathbb{E}[\chi_j^2] \approx 1 + N \sum_{k} r_{jk}^2 \mathbb{E}[h_k^2]$$

We proceed in three steps:
1. **Format the base GWAS summary statistics (`Height.summaries`)**: Just like in `GWAS7`, we keep well-imputed common variants (`MAF >= 0.01` and `INFO >= 0.8`) and output the 6 columns required by `LDAK`: `Predictor` (SNP ID), `A1` (effect allele), `A2` (other allele), `Direction` (`+1` or `-1`), `P` ($p$-value), and `n` (sample size). *(You do not need to memorize the `awk` formatting snippet below—it simply reformats the columns of `Data/Height.QC.gz`.)*
2. **Compute the LD tagging file (`ldak --calc-tagging`)**: Calculates how much heritability each SNP tags across a `1000 kb` window in our European reference panel (`Data/EUR.QC`), and saves the heritability matrix (`--save-matrix YES`).
3. **Estimate heritability (`ldak --sum-hers`)**: Regresses the GWAS summary statistics (`--summary`) on the LD tagging file (`--tagfile`) to estimate total SNP heritability (`sumher_ldak.hers`) and each individual SNP's expected heritability (`sumher_ldak.ind.hers`).

In [ ]:
# 1. Format Data/Height.QC.gz into LDAK summary statistics format (Predictor A1 A2 Direction P n)
gzip -dc Data/Height.QC.gz | awk '
NR==1 {
    for (i=1; i<=NF; i++) col[$i]=i
    print "Predictor A1 A2 Direction P n"
    next
}
$col["MAF"] >= 0.01 && $col["INFO"] >= 0.8 && !seen[$col["SNP"]]++ {
    eff = ("OR" in col) ? log($col["OR"]) : $col["BETA"]
    dir = (eff >= 0) ? 1 : -1
    p   = $col["P"]
    n   = ("N" in col) ? $col["N"] : 253288
    if (p > 0 && p <= 1) print $col["SNP"], $col["A1"], $col["A2"], dir, p, n
}' > Results/GWAS8/Height.summaries

head -n 5 Results/GWAS8/Height.summaries

# 2. Calculate tagging file and heritability matrix under the LDAK-Thin model (power = -0.25)
ldak --calc-tagging Results/GWAS8/tag_ldak \
     --bfile Data/EUR.QC \
     --weights Results/GWAS8/weights.thin \
     --power -0.25 \
     --window-kb 1000 \
     --save-matrix YES

# 3. Estimate SNP heritability from summary statistics
ldak --sum-hers Results/GWAS8/sumher_ldak \
     --summary Results/GWAS8/Height.summaries \
     --tagfile Results/GWAS8/tag_ldak.tagging \
     --matrix Results/GWAS8/tag_ldak.matrix \
     --check-sums NO

Let's inspect the heritability estimates from all three analyses. When reading `.reml` and `.hers` files, focus on:
- **`Her_All` (`Her` and `SD`)**: The estimated total SNP heritability ($\hat{h}^2_{\text{SNP}}$) and its standard error (`SD`).
- **`Scaling` (in `.hers`)**: Measures the intercept of the summary-statistic regression (`1.0` indicates no inflation from population stratification or cryptic relatedness).

In [ ]:
echo "=== 1. Individual-level REML: GCTA Model (power = -1) ==="
cat Results/GWAS8/reml_gcta.reml

echo ""
echo "=== 2. Individual-level REML: LDAK-Thin Model (power = -0.25) ==="
cat Results/GWAS8/reml_ldak.reml

echo ""
echo "=== 3. Summary-statistics SumHer: LDAK-Thin Model (N = 253,288) ==="
cat Results/GWAS8/sumher_ldak.hers

### Visualizing Per-SNP Heritability in R

<img src="Images/R.png" alt="R" width="40"> Switch to the R kernel.

In `R`, we can visualize the distribution of per-SNP expected heritabilities (`sumher_ldak.ind.hers`) produced by `ldak --sum-hers`:

In [ ]:
options(warn = -1)
options(jupyter.plot_mimetypes = 'image/png')
options(repr.plot.width = 9, repr.plot.height = 4)

suppressMessages(suppressWarnings(library(ggplot2)))

ind_hers <- read.table("Results/GWAS8/sumher_ldak.ind.hers", header = TRUE)
colnames(ind_hers) <- c("Predictor", "Heritability")

p_her <- ggplot(subset(ind_hers, Heritability > 0), aes(x = Heritability)) +
  geom_histogram(bins = 40, col = "black", fill = "tomato") +
  labs(title = "Distribution of Per-SNP Expected Heritabilities (LDAK-Thin, power = -0.25)",
       x = "Per-SNP Expected Heritability",
       y = "Number of SNPs") +
  theme_bw() +
  theme(axis.title = element_text(size = 14), axis.text = element_text(size = 12))

show(p_her)
cat("Total SNP heritability (sum across SNPs):", round(sum(ind_hers$Heritability), 4), "\n")

:::{.callout-important title="Stop - Read - Solve"}

Examine the output of `reml_gcta.reml`, `reml_ldak.reml`, and `sumher_ldak.hers` above:

- **Q1. How many SNPs in `ind_hers` have `Heritability > 0` (retained after `--thin`) compared to the total number of SNPs (`nrow(ind_hers)`)?**
- **Q2. Compare the standard error (`SD`) of `Her_All` in the individual-level REML analyses (`reml_ldak.reml`, where $N \approx 500$) against the summary-statistics SumHer analysis (`sumher_ldak.hers`, where $N = 253,288$).** Why is the SumHer standard error so much smaller?
- **Q3. Plot or compute the relationship between LD tagging (`Results/GWAS8/tag_ldak.tagging`) and GWAS $\chi^2$ test statistics (`Results/GWAS8/Height.summaries`) in `R`:** Why does a positive slope between LD tagging and $\chi^2$ indicate true polygenic heritability rather than population stratification?

Hint: Use `sum(ind_hers$Heritability > 0)` and `nrow(ind_hers)` in `R`, and inspect `Results/GWAS8/reml_ldak.reml` vs. `Results/GWAS8/sumher_ldak.hers`.

:::

In [ ]:
# Write your code here


:::{.callout-tip title="Solution" collapse="true"}

In `R`, we can compare the retained SNPs (`Heritability > 0`), inspect the `Her_All` estimates and standard errors (`SD`), and compute the correlation between LD tagging and marginal GWAS $\chi^2$ statistics:

In [ ]:
# Q1. Compare retained SNPs (Heritability > 0) vs. total SNPs
cat("Total SNPs in EUR.QC:    ", nrow(ind_hers), "\n")
cat("Retained after thinning: ", sum(ind_hers$Heritability > 0), "\n")
cat("Pruned duplicate SNPs:   ", sum(ind_hers$Heritability == 0), "\n\n")

# Q2. Compare Her_All and SD across individual-level REML (N ~ 500) and SumHer (N = 253,288)
reml_lines   <- grep("^Component|^Her_", readLines("Results/GWAS8/reml_ldak.reml"), value = TRUE)
sumher_lines <- grep("^Component|^Her_", readLines("Results/GWAS8/sumher_ldak.hers"), value = TRUE)
reml_ldak    <- read.table(text = reml_lines, header = TRUE)
sumher_ldak  <- read.table(text = sumher_lines, header = TRUE)
cat("=== Individual-level REML (LDAK-Thin) ===\n")
print(subset(reml_ldak, Component == "Her_All"))
cat("\n=== Summary-statistics SumHer (LDAK-Thin) ===\n")
print(subset(sumher_ldak, Component == "Her_All"))

# Q3. Inspect and plot the relationship between LD tagging and GWAS chi-square test statistics
tagging <- read.table("Results/GWAS8/tag_ldak.tagging", header = TRUE, fill = TRUE)
sumstat <- read.table("Results/GWAS8/Height.summaries", header = TRUE)
m_tag   <- merge(tagging[, c("Predictor", "Tagging")], sumstat[, c("Predictor", "P")], by = "Predictor")
m_tag$Tagging <- as.numeric(m_tag$Tagging)
m_tag$Chisq   <- qchisq(m_tag$P, df = 1, lower.tail = FALSE)
cat("\nCorrelation between LD Tagging and GWAS Chi-square:",
    round(cor(m_tag$Tagging, m_tag$Chisq, use = "complete.obs"), 4), "\n")

# Group SNPs into 20 LD-tagging bins to visualize the upward SumHer slope
m_tag$Bin <- cut(m_tag$Tagging, breaks = quantile(m_tag$Tagging, probs = seq(0, 1, length.out = 21), na.rm = TRUE), include.lowest = TRUE)
bin_df <- aggregate(cbind(Tagging, Chisq) ~ Bin, data = m_tag, FUN = mean)
p_sumher <- ggplot(bin_df, aes(x = Tagging, y = Chisq)) +
  geom_point(size = 3, color = "steelblue") +
  geom_smooth(method = "lm", se = FALSE, color = "tomato", linewidth = 1.1) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "grey40") +
  theme_bw(base_size = 13) +
  labs(
    title = "SumHer Intuition: Mean GWAS Chi-Square vs. LD Tagging Score (20 Quantile Bins)",
    x = "Mean LD Tagging Score (External EUR.QC Panel)",
    y = expression("Mean Observed GWAS " ~ chi^2)
  )
show(p_sumher)

- **Q1**: Comparing `nrow(ind_hers)` and `sum(ind_hers$Heritability > 0)` shows how many redundant SNPs ($r^2 > 0.98$ within 100 kb) were pruned (`Heritability = 0` in `sumher_ldak.ind.hers`).
- **Q2**: In `Data/EUR.QC` ($N \approx 500$), the individual-level REML standard error is large ($\text{SD}(\hat{h}^2) \approx \frac{316}{N} \approx 0.6$), because $500$ unrelated individuals provide limited information on subtle genome-wide allele sharing. By contrast, `sumher_ldak.hers` uses summary statistics from $N = 253,288$ individuals, yielding a much smaller standard error.
- **Q3**: Under a polygenic trait, SNPs that tag more heritability through LD have systematically higher expected $\chi^2$ test statistics (the **slope** against the tagging values estimates $h^2_{\text{SNP}}$). Pure confounding or cryptic relatedness inflates $\chi^2$ statistics uniformly across all SNPs regardless of LD tagging (shifting the **intercept** away from $1$).

:::

---

### Conclusion

In this tutorial, we have learned how SNP heritability ($h^2_{\text{SNP}}$) is defined, how the GCTA ($\alpha = -1$) and LDAK ($\alpha = -0.25$) heritability models differ in their assumptions about allele frequency and LD, and how to estimate $h^2_{\text{SNP}}$ from both individual-level genotypes (`ldak --reml`) and GWAS summary statistics (`ldak --sum-hers`). In the upcoming sessions, we will see how the individual-level polygenic variance model ($\sigma_g^2 K$) leads directly into **Linear Mixed Model GWAS (`GWAS9`)**, while pairing GWAS summary statistics with an **external LD reference panel** (`sumher_ldak.ind.hers`) powers post-GWAS fine-mapping (`GWAS10`) and Bayesian polygenic prediction (`GWAS11`).

:::{.callout-note title="Other take-home messages"}

- **Heritability bounds polygenic prediction**: A polygenic score built from common SNPs cannot explain more variance in an independent target sample than the trait's SNP heritability $h^2_{\text{SNP}}$.
- **Sample size requirements**: Individual-level REML (`ldak --reml`) requires $N \ge 3,000\text{–}5,000$ unrelated individuals for small standard errors, whereas summary-statistics SumHer (`ldak --sum-hers`) leverages large published meta-analyses ($N > 50,000$) paired with a smaller LD reference panel.
- **Observed vs. Liability-Scale Heritability for Binary Traits**: When estimating heritability for a binary case-control trait, always supply `--prevalence <K>` (the population disease prevalence) to `ldak --reml` or `ldak --sum-hers` to convert the estimate from the observed (0/1) scale to the continuous **liability scale**.

:::